In [ ]:
!pip install torch torchvision einops scikit-learn numpy opencv-python tqdm kaggle

In [ ]:
import os
import zipfile

# 1. Secure and set up the Kaggle API configuration environment
os.makedirs('/root/.kaggle', exist_ok=True)

if os.path.exists('kaggle.json'):
    # Move uploaded token file to the official secure path location
    !mv kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json
    print("Kaggle API Credentials successfully configured.")
else:
    print("Warning: 'kaggle.json' file not detected in the current workspace directory.")
    print("Please upload your API token file before continuing.")

# 2. Programmatically trigger the background Celeb-DF v2 dataset download
print("\nDownloading Celeb-DF v2 Dataset via Kaggle API CLI (This may take several minutes)...")
!kaggle datasets download -d herbwood/celebdfv2

# 3. Clean extraction pipeline routines
ZIP_FILE_PATH = "celebdfv2.zip"
EXTRACTION_TARGET_DIR = "./celeb-df-v2"

if os.path.exists(ZIP_FILE_PATH):
    print("\nExtracting archive contents, please stand by...")
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACTION_TARGET_DIR)
    print(f"Extraction complete! Files extracted directly to: {EXTRACTION_TARGET_DIR}")

    # Clean up zip artifact to free system disk space
    os.remove(ZIP_FILE_PATH)
else:
    print(f"Error: Target archive {ZIP_FILE_PATH} could not be located.")

In [ ]:
import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from einops import rearrange, repeat
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm
import cv2
from PIL import Image
from torchvision import transforms

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Pipeline initialized. Running on backend device: {device}")

In [ ]:
class TexturalFeatureExtractor(nn.Module):
    """
    Shallow Conv Net inspired by Xception entry flow layers.
    Focuses on capturing structural and phase inconsistencies across deepfake regions.
    """
    def __init__(self, in_channels=3, out_channels=128):
        super().__init__()
        # Corrected syntax corruption typo
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, out_channels, kernel_size=3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Expects: [B * T, Channels, Height, Width]
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))
        return x  # Outputs downsampled latent map: [B * T, out_channels, H_new, W_new]

class CelebDFVideoDataset(Dataset):
    """
    Reads Celeb-DF folders directly and partitions them programmatically
    into deterministic train/validation splits by index.
    """
    def __init__(self, root_dir, split='train', split_ratio=0.8, num_frames=6, img_size=128, seed=42):
        self.num_frames = num_frames
        self.img_size = img_size
        self.samples = []

        # Standardization transform pipeline
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        # Explicit directory targeting matching the Celeb-DF dataset structure
        categories = {
            'Celeb-real': 0.0,
            'Celeb-synthesis': 1.0
        }

        for folder_name, class_tag in categories.items():
            class_dir = os.path.join(root_dir, folder_name)
            if not os.path.exists(class_dir):
                print(f"Warning Partition Mapping: Directory target missing -> {class_dir}")
                continue

            # Aggregate video files safely matching all major extensions
            folder_videos = []
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                    folder_videos.append((os.path.join(class_dir, fname), class_tag))

            # Sort natively to guarantee absolute alignment across separate data worker runs
            folder_videos.sort()

            # Deterministically shuffle entries using a distinct random state generator
            rng = np.random.default_rng(seed)
            rng.shuffle(folder_videos)

            # Map boundary split constraints
            split_idx = int(len(folder_videos) * split_ratio)
            if split == 'train':
                self.samples.extend(folder_videos[:split_idx])
            else:
                self.samples.extend(folder_videos[split_idx:])

    def __len__(self):
        return len(self.samples)

    def _load_video(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames <= 0:
            return torch.zeros(self.num_frames, 3, self.img_size, self.img_size)

        # Uniform mathematical step selection mapping across the full video lifespans
        indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
        frames = []

        # High-performance Fast Seek frame decoding optimization step
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil_img = Image.fromarray(frame)
                frames.append(self.transform(pil_img))
            else:
                break

        cap.release()

        # Handle fallback framing bounds for clipped/corrupt media files
        while len(frames) < self.num_frames:
            frames.append(frames[-1] if len(frames) > 0 else torch.zeros(3, self.img_size, self.img_size))

        return torch.stack(frames[:self.num_frames])

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        try:
            video_tensor = self._load_video(video_path)
        except Exception as e:
            print(f"Error reading {video_path}: {e}")
            video_tensor = torch.zeros(self.num_frames, 3, self.img_size, self.img_size)

        return video_tensor, torch.tensor([label], dtype=torch.float32)

print("Feature extraction backbone and Celeb-DF dataset handlers successfully declared.")

In [ ]:
class DecomposedSpatialTemporalAttention(nn.Module):
    """
    Factorizes full spatial-temporal attention matrices into decoupled dimensions.
    Integrates the Self-Subtract operation to highlight raw inter-frame motion residuals.
    """
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        # Temporal attention query, key, value maps
        self.q_temp = nn.Linear(dim, dim, bias=qkv_bias)
        self.k_temp = nn.Linear(dim, dim, bias=qkv_bias)
        self.v_temp = nn.Linear(dim, dim, bias=qkv_bias)

        # Spatial attention query, key, value maps
        self.q_spatial = nn.Linear(dim, dim, bias=qkv_bias)
        self.k_spatial = nn.Linear(dim, dim, bias=qkv_bias)
        self.v_spatial = nn.Linear(dim, dim, bias=qkv_bias)

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        # x input structure: [Batch, T + 1, HW + 1, Channels]
        B, T_plus_1, HW_plus_1, C = x.shape

        # --- STEP 1: SELF-SUBTRACT MECHANISM ---
        x_sub = torch.cat([
            x[:, 0:2, :, :],
            x[:, 2:, :, :] - x[:, 1:-1, :, :]
        ], dim=1)

        # --- STEP 2: TEMPORAL ATTENTION ROUTING ---
        q_t = self.q_temp(x_sub)
        k_t = self.k_temp(x_sub)
        v_t = self.v_temp(x)  # Identity mapping tracking vector

        q_t = rearrange(q_t, 'b t s (h d) -> b h s t d', h=self.num_heads)
        k_t = rearrange(k_t, 'b t s (h d) -> b h s t d', h=self.num_heads)
        v_t = rearrange(v_t, 'b t s (h d) -> b h s t d', h=self.num_heads)

        attn_t = (q_t @ k_t.transpose(-2, -1)) * self.scale
        attn_t = attn_t.softmax(dim=-1)
        attn_t = self.attn_drop(attn_t)

        out_t = attn_t @ v_t
        out_t = rearrange(out_t, 'b h s t d -> b t s (h d)')

        # --- STEP 3: SPATIAL ATTENTION ROUTING ---
        q_s = self.q_spatial(out_t)
        k_s = self.k_spatial(out_t)
        v_s = self.v_spatial(out_t)

        q_s = rearrange(q_s, 'b t s (h d) -> b h t s d', h=self.num_heads)
        k_s = rearrange(k_s, 'b t s (h d) -> b h t s d', h=self.num_heads)
        v_s = rearrange(v_s, 'b t s (h d) -> b h t s d', h=self.num_heads)

        attn_s = (q_s @ k_s.transpose(-2, -1)) * self.scale
        attn_s = attn_s.softmax(dim=-1)
        attn_s = self.attn_drop(attn_s)

        out_s = attn_s @ v_s
        out_s = rearrange(out_s, 'b h t s d -> b t s (h d)')

        # Final layout projection
        out = self.proj(out_s)
        out = self.proj_drop(out)
        return out

class Mlp(nn.Module):
    """Standard Multilayer Perceptron mapping for feed-forward transformer layers"""
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features * 4
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))

class ISTVTTransformerBlock(nn.Module):
    """Unified Decomposed Spatial-Temporal Encoder Block wrapper"""
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = DecomposedSpatialTemporalAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio), drop=drop)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
class ISTVT(nn.Module):
    """
    The fully assembled Interpretable Spatial-Temporal Video Transformer network
    """
    def __init__(self, num_frames=6, img_size=128, patch_size=1, embed_dim=128, depth=4, num_heads=4):
        super().__init__()
        self.num_frames = num_frames

        # Instantiate convolutional entry front-end
        self.feature_extractor = TexturalFeatureExtractor(in_channels=3, out_channels=embed_dim)

        # Post downsampling metrics calculations (stride 2 x 3 operations -> 128 / 8 = 16)
        self.feature_map_hw = img_size // 8
        num_patches = (self.feature_map_hw // patch_size) ** 2

        # State categorization classification parameters instantiation
        self.spatial_cls_token = nn.Parameter(torch.zeros(1, 1, 1, embed_dim))
        self.temporal_cls_token = nn.Parameter(torch.zeros(1, 1, num_patches + 1, embed_dim))

        # Positional coordinate configuration metrics
        self.pos_embed = nn.Parameter(torch.zeros(1, num_frames + 1, num_patches + 1, embed_dim))

        self.blocks = nn.ModuleList([
            ISTVTTransformerBlock(dim=embed_dim, num_heads=num_heads)
            for _ in range(depth)
        ])

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 1) # Raw prediction logit mapping for direct BCE loss
        )

        nn.init.trunc_normal_(self.spatial_cls_token, std=0.02)
        nn.init.trunc_normal_(self.temporal_cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B, T, C, H, W = x.shape

        # Flatten sequence dimension lengths processing steps
        x = rearrange(x, 'b t c h w -> (b t) c h w')
        features = self.feature_extractor(x)

        tokens = rearrange(features, '(b t) c h w -> b t (h w) c', b=B, t=T)

        # 1) Spatial token insertions
        spatial_cls = self.spatial_cls_token.expand(B, T, -1, -1)
        tokens = torch.cat([spatial_cls, tokens], dim=2)

        # 2) Temporal token insertions
        temporal_cls = self.temporal_cls_token.expand(B, -1, -1, -1)
        tokens = torch.cat([temporal_cls, tokens], dim=1)

        # Embed positional metrics coordinates
        tokens = tokens + self.pos_embed

        for block in self.blocks:
            tokens = block(tokens)

        # Pull critical classification decision token coordinates
        cls_prediction_token = tokens[:, 0, 0, :]
        logits = self.mlp_head(cls_prediction_token)
        return logits

print("ISTVT deepfake network blueprint architecture compiled successfully.")

In [ ]:
# --- Path Configurations ---
# Pointing directly to the extracted target directory from Cell 2
DATASET_ROOT = "./celeb-df-v2"

NUM_FRAMES = 6
IMG_SIZE = 128
BATCH_SIZE = 8   # Adjust to 4 or 2 if you run out of GPU Memory (OOM)
EPOCHS = 8
LEARNING_RATE = 1e-4

print("Processing automated train/val partitioning loops over Celeb-DF directories...")
train_dataset = CelebDFVideoDataset(root_dir=DATASET_ROOT, split='train', split_ratio=0.8, num_frames=NUM_FRAMES, img_size=IMG_SIZE)
val_dataset = CelebDFVideoDataset(root_dir=DATASET_ROOT, split='val', split_ratio=0.8, num_frames=NUM_FRAMES, img_size=IMG_SIZE)

# Configured parallel worker threads and memory pinning to maximize hardware performance
# (If your loop hangs indefinitely, set num_workers=0)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = val_loader

print(f"Dataset configurations mapped: {len(train_dataset)} training steps, {len(val_dataset)} validation evaluation files.")

print("Instantiating network weights and AdamW optimization routines...")
model = ISTVT(num_frames=NUM_FRAMES, img_size=IMG_SIZE, depth=4, num_heads=4).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)

# --- Training Pipeline Loop Execution ---
print("\nInitiating training iteration loops...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    start_time = time.time()

    if len(train_loader) == 0:
        print(f"\nExecution Stopped: No target files located inside path: {DATASET_ROOT}")
        print("Please check that Celeb-real and Celeb-synthesis folders exist inside that path directory.")
        break

    for step, (videos, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")):
        videos, labels = videos.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(videos)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * videos.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    elapsed = time.time() - start_time
    print(f"Epoch [{epoch}/{EPOCHS}] Complete -> Cross-Entropy Loss: {epoch_loss:.4f} | Loop Duration: {elapsed:.2f}s")

# --- Inference Analysis Phase ---
print("\nRunning tracking evaluations across test verification arrays...")
model.eval()
all_preds = []
all_labels = []

if len(test_loader) > 0:
    with torch.no_grad():
        for videos, labels in tqdm(test_loader, desc="Inference Processing"):
            videos = videos.to(device)
            logits = model(videos)
            probs = torch.sigmoid(logits).cpu().numpy()

            all_preds.extend(probs)
            all_labels.extend(labels.numpy())

    all_preds = np.array(all_preds).reshape(-1)
    all_labels = np.array(all_labels).reshape(-1)
    binary_preds = (all_preds >= 0.5).astype(np.float32)

    accuracy = accuracy_score(all_labels, binary_preds)
    try:
        auc_score = roc_auc_score(all_labels, all_preds)
    except ValueError:
        auc_score = 0.5

    print("\n" + "="*50)
    print("         CELEB-DF PIPELINE MODEL METRICS          ")
    print("="*50)
    print(f" Model Accuracy Assessment   : {accuracy * 100:.2f}%")
    print(f" Receiver Operating Area AUC : {auc_score:.4f}")
    print("="*50)